In [ ]:
!pip install unsloth
!pip install transformers datasets accelerate peft trl
!pip install evaluate rouge_score nltk

In [ ]:
import pandas as pd
import torch
import math
import sqlite3
from datetime import datetime

from datasets import Dataset
from unsloth import FastLanguageModel
from transformers import TrainingArguments
from trl import SFTTrainer

import evaluate

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
df = pd.read_csv("BengaliEmpatheticConversationsCorpus .csv")

df.head()

In [ ]:
df = df[['Questions','Answers']]

df = df.dropna()

df = df.rename(columns={
    "Questions":"input",
    "Answers":"response"
})

print(df.shape)

In [ ]:
def format_prompt(row):

    return f"""### Instruction:
{row['input']}

### Response:
{row['response']}"""

df["text"] = df.apply(format_prompt, axis=1)

df.head()

In [ ]:
dataset = Dataset.from_pandas(df[['text']])

print(dataset)

In [ ]:
dataset = dataset.train_test_split(test_size=0.1)

train_dataset = dataset["train"]
test_dataset = dataset["test"]

print(train_dataset)
print(test_dataset)

In [ ]:
max_seq_length = 2048

model, tokenizer = FastLanguageModel.from_pretrained(

    model_name="unsloth/llama-3-8b-bnb-4bit",
    max_seq_length=max_seq_length,
    load_in_4bit=True

)

In [ ]:
model = FastLanguageModel.get_peft_model(

    model,

    r=16,

    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ],

    lora_alpha=16,
    lora_dropout=0,
    bias="none",

    use_gradient_checkpointing=True
)

In [ ]:
trainer = SFTTrainer(

    model=model,
    tokenizer=tokenizer,

    train_dataset=train_dataset,
    eval_dataset=test_dataset,

    dataset_text_field="text",
    max_seq_length=2048,

    args=TrainingArguments(

        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,

        warmup_steps=5,

        max_steps=100,

        learning_rate=2e-4,

        logging_steps=10,

        fp16=True,

        output_dir="outputs",

        optim="adamw_8bit"
    )
)

In [ ]:
trainer.train()

In [ ]:
model.save_pretrained("bengali_empathetic_llama3")

tokenizer.save_pretrained("bengali_empathetic_llama3")

In [ ]:
FastLanguageModel.for_inference(model)

prompt = """
### Instruction:
আমি ধূমপানে আসক্ত। আমি কিভাবে থামাতে পারি?

### Response:
"""

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

outputs = model.generate(

    **inputs,
    max_new_tokens=120

)

print(tokenizer.decode(outputs[0]))

In [ ]:
bleu = evaluate.load("bleu")

rouge = evaluate.load("rouge")

In [ ]:
preds = ["তুমি একা নও। আমি তোমার পাশে আছি।"]

refs = [["তুমি একা নও। আমি বুঝতে পারছি তুমি কষ্ট পাচ্ছ।"]]

bleu_score = bleu.compute(predictions=preds, references=refs)

rouge_score = rouge.compute(predictions=preds, references=refs)

print("BLEU:", bleu_score)
print("ROUGE:", rouge_score)

In [ ]:
eval_loss = 1.5

perplexity = math.exp(eval_loss)

print("Perplexity:", perplexity)

In [ ]:
conn = sqlite3.connect("experiments.db")

cursor = conn.cursor()

cursor.execute("""

CREATE TABLE IF NOT EXISTS LLAMAExperiments(

id INTEGER PRIMARY KEY,

model_name TEXT,

lora_config TEXT,

train_loss REAL,

val_loss REAL,

metrics TEXT,

timestamp TEXT

)

""")

In [ ]:
cursor.execute("""

INSERT INTO LLAMAExperiments

(model_name,lora_config,train_loss,val_loss,metrics,timestamp)

VALUES (?,?,?,?,?,?)

""",

(

"llama3-8b",

"r16_alpha16",

1.2,

1.5,

"BLEU/ROUGE",

str(datetime.now())

))

conn.commit()

In [ ]:
cursor = conn.cursor()

In [ ]:
cursor.execute("SELECT * FROM LLAMAExperiments")

rows = cursor.fetchall()

for row in rows:
    print(row)